# Les Générateurs et Coroutines

Dans cette section, nous allons explorer les **générateurs**, une fonctionnalité puissante de Python permettant de travailler avec des flux potentiellement **infini de données** et de créer des **Coroutines**.

## Création d'un premier générateur

Pour créer un générateur, c'est tout simple : il suffit de créer une fonction classique, mais au lieu d'utiliser un `return` on utilise à la place Le mot-clé **`yield`**, ce qui permet à notre fonction de renvoyer une valeur et de "mettre en pause" son exécution jusqu'à son prochain appel, reprenant ainsi là où elle s’était précédemment arrêtée. (créant ainsi une **coroutine**)




### Syntaxe
```python
def generateur():
    yield valeur
```

Pour appeler le générateur, il suffit de le faire passer dans la fonction `next()`

### Exemple Simple

In [ ]:
def mon_generateur():
    yield "Bonjour"

In [ ]:
gen = mon_generateur()

In [ ]:
next(gen)

In [ ]:
next(gen, "Terminé")

On peut ainsi empiler des `yield`. A chaque `yield` rencontré, la coroutine s'arretera jusqu'a etre a nouveau appelée

In [ ]:
def mon_generateur():
    n = 0
    print("C'est le début de notre fonction")
    yield n
    n = 10
    print("On reprend le travail")
    yield n
    a = - 1
    print("cette fois-ci nous avons ouvert un fichier")
    yield n - a
    yield "Opération 4"

In [ ]:
gen = mon_generateur()

In [ ]:
next(gen)

#### Dans la pratique, combinaison avec While

Les `yield` sont souvent encapsulés dans des boucles `while`, afin d'executer des instructions de code, retourner un résultat, et attendre le prochain appel pour reprendre un cycle de calcul.

(*Rappelez-vous qu'en général, une bonne fonction se veut "courte" et "mono-fonction"*)


Voici par exemple un générateur qui produit des nombres a l'infini

In [ ]:
def number_generator():
    n = 0
    while True:
        yield n
        n += 1


In [ ]:
# Utilisation
gen = number_generator()


In [ ]:
next(gen)

### Faire passer des parametres dans un Générateur

Comme toute fonction, un générateur peut prendre en entrée des parametres, ainsi que des *args, et **kwargs, afin de rendre son fonctionnement plus dynamique

In [ ]:
def nombres_pairs(maximum: int):
    """Génère des nombres jusqu’à atteindre un maximum."""
    n = 0
    while n < maximum:
        yield n
        n += 2

In [ ]:
gen = nombres_pairs(10)

In [ ]:
next(gen)

### Terminer poprement un générateur

Comme nous pouvons le voir, une fois le bout de son programme atteint un générateur nous retourne l'Exception `StopIteration`.

Pour "terminer proprement" un générateur, il convient donc d'emballer notre code au sein d'une structure `try` `except`, qui se combine avec une commande `return` qui nous donne acces a la derniere valeur du génerateur

In [ ]:
def nombres_pairs(maximum: int):
    """Génère des nombres jusqu’à atteindre un maximum."""
    n = 0
    while n < maximum:
        yield n
        n += 2
    return n

In [ ]:
gen = nombres_pairs(10)

In [ ]:
try:
    while True:
        print(next(gen))
except StopIteration as e:
    print("valeur finale:", e.value)

#### Interromptre manuellement un Génerateur

Si nous voulons interompre manuellement un générateur, on peut le faire avec une structure `try` / `finally` ainsi que la commande `close()`.

In [ ]:
def gen():
    n = 0
    try:
        while True:
            yield n
            n += 1
    finally:
        print("Fermeture du générateur")
        return n

g = gen()

In [ ]:
next(g)

In [ ]:
g.close()  # met fin au générateur

#### L'utilisation des boucles For

Si le générateur a une fin, alors on peut utiliser une boucle `for` pour l'executer jusqu'a son terme, sans risque de rencontrer une Exception `StopIteration`

In [ ]:
gen = nombres_pairs(10)

In [ ]:
for x in nombres_pairs(10):
    print(x)

# Comment faire passer des valeurs externes dans un générateur ?

A présent, comment faire pour reprendre l'execution de notre coroutine, en y passant une valeur externe ? la réponse -> utiliser la commande `send()`.


En effet, Nous avons vu la commande `next()` qui permet d'“avancer” un générateur. `send()` fait pareil, mais en envoyant en plus une valeur dans le générateur.


### Syntaxe:
quand on écrit :

```python
gen.send(x)
```

→ Python reprend l’exécution du générateur là où il s’était arrêté sur un yield,

→ injecte la valeur x dans l’expression yield,

→ et retourne la valeur du prochain yield


Démonstration : 

In [ ]:
def echo():
    while True:
        value = yield  # valeur recu lors du prochain appel avec send()
        print(f"Received: {value}")

# création du générateur
generator = echo()

# démarrage
next(generator)

In [ ]:
generator.send("Bonjour")

In [ ]:
next(generator)

#### Un exemple pratique: 
Définition d'une coroutine qui calcule une somme cumulative.
- `yield total` renvoie la valeur actuelle du total et suspend l'exécution.
- `send(valeur)` permet d'envoyer une valeur à la coroutine qui sera assignée à valeur.
- La coroutine reprend exactement là où elle s’était arrêtée après le `yield`.

Dans ce mode d'execution, il est nécessaire de démarrer avec la commande `next()`

In [ ]:
def accumulator():
    total = 0
    while True:
        value = yield total # recoit une nouvelle valeur en entrée (value) et produit le `total` en sortie
        total += value # modifie la valeur `total` vivant dans le générateur

In [ ]:
acc = accumulator()
next(acc)

In [ ]:
print(acc.send(-3))

In [ ]:
print(acc.send(10))

In [ ]:
print(acc.send(20))

## Utiliser des générateurs... dans des générateurs

Parfois, on peut vouloir placer des générateurs dans d'autres générateurs...

```python
def couper_oignons():
    ...

def couper_tomates():
    ...

def couper_champignons():
    ...

def parser():
    yield from couper_oignons()
    yield from couper_tomates()
    yield from couper_champignons()
```

la commande `yield from` permet de déléguer la génération de valeurs d’un générateur à un autre générateur.

Cela simplifie le code quand on veut combiner ou chaîner plusieurs générateurs

##### Sans `yield from`

In [ ]:
def gen1():
    yield "gen1 - opération 1"
    yield "gen1 - opération 2"
    yield "gen1 - opération 3"

In [ ]:
def gen2():
    yield "gen2 - opération 1"
    yield "gen2 - opération 2"
    yield "gen2 - opération 3"

In [ ]:
def combined():
    for value in gen1():
        yield value
    for value in gen2():
        yield value

In [ ]:
gen = combined()

In [ ]:
next(gen)

#### Avec `yield from`

In [ ]:
def combined():
    yield from gen1()
    yield from gen2()

In [ ]:
gen = combined()

In [ ]:
next(gen)

En plus de rendre le code plus lisible, la commande `yield from` permet également de transmettre d'un générateur a l'autre les exceptions, et de déléguer les commandes `next()`, `send()`, `throw()` et `close()`.

#### Exemple

Créons un générateur simulant la lecture d'un fichier.

In [ ]:
def reader():
    """Un générateur qui simule la lecture d'un fichier ou bien d'un port d'entrée..."""
    for i in range(4):
        yield i

In [ ]:
def reader_wrapper(generateur):
    # On itere a travers les résultats du générateur
    for v in generateur:
        yield v

In [ ]:
wrap = reader_wrapper(reader())
for i in wrap:
    print(i)

A présent, créons une coroutine visant a écrire dans un fichier.

In [ ]:
def writer():
    """Une coroutine qui écrit des valuers numériques dans un fichier"""
    while True:
        w = (yield)
        print(w)

In [ ]:
def writer_wrapper(coroutine):
    coroutine.send(None)
    while True:
        try:
            x = (yield)  # Capture la valeur envoyée par send()
            coroutine.send(x)  # transfmet la valeur a la coroutine
        except StopIteration:
            pass

In [ ]:
def writer_wrapper(coroutine):
    yield from coroutine

In [ ]:
wrap = writer_wrapper(writer())
next(wrap)

In [ ]:
wrap.send("ReBonjour")

# Applications des Générateurs et des Coroutines

- **Flux de donnés volumineux / infini**: les générateurs peuvent etre bien plus efficaces que les Listes/Tuples et autres collections afin de gérer des volumes de données importants. En effet, ils évitent de stocker les données dans la RAM, car il suffit de les générer les unes apres les autres... Donnant ainsi lieu a des programmes tres maigres et performants.

- **Pipelines** : Dans les pipelines de traitement de données, la fonction `send` peut être utilisée pour transmettre des données entre différentes étapes du pipeline, permettant ainsi des conceptions modulaires et flexibles. On peut ainsi créer des coroutines qui réalisent des tâches complexes en recevant des valeurs à des points spécifiques de leur exécution.

- **Gestion d’état** : Vous pouvez utiliser `send` pour gérer l’état dans votre application en permettant au générateur de réagir à des événements ou commandes externes, contrôlant ainsi son comportement de manière dynamique.

### Exemple 1 : Flux Volumineux de données
Lisons un grand fichier ligne par ligne.

In [ ]:
def lines_generator(file_path):
    """Générateur qui lit un fichier ligne par ligne."""
    with open(file_path, 'r') as f:
        for line in f:
            yield line


In [ ]:
chemin = "../01 Cours débutant/data/exercice.txt"

file_gen = lines_generator(chemin)

In [ ]:
next(file_gen)

Pour améliorer ce code (et éviter le `StopItération`) Boucle for

In [ ]:
for x in lines_generator(chemin):
    print(x)

### Imaginons maintenant vouloir faire cela sur des millions de fichiers

Pour ca, on va utiliser plusieurs générateurs, et les appeler avec `yield from`

In [ ]:
import os

def files_generator(folder_path="data/generators/"):
    """Générateur qui fournit les fichiers à lire dans un dossier."""
    for entry in os.scandir(folder_path):
        if entry.is_file():   # évite les dossiers
            yield entry.path

In [ ]:
def lines_generator(file_path):
    """Générateur qui lit un fichier ligne par ligne."""
    with open(file_path, 'r') as f:
        for line in f:
            yield line

In [ ]:
def main_generator():
    """Générateur qui parcourt tous les fichiers et yield leurs lignes."""
    for file_path in files_generator():
        # Délègue la génération de lignes à lines_generator
        yield from lines_generator(file_path)

In [ ]:
# Utilisation
for line in main_generator():
    print(line.strip())

### Exemple 2 : Flux Infini de données
Générons une suite infinie de nombres de Fibonacci.

In [ ]:
def fibonacci():
    """Génère une suite infinie de nombres de Fibonacci."""
    a, b = 0, 1
    while True:
        yield a
        a, b = b, a + b


In [ ]:
gen = fibonacci()

In [ ]:
next(gen)

In [ ]:
for _ in range(10):
    print(next(gen), end=" ") 

#### Comparons la taille en RAM de ce générateur par rapport a une liste Python de 100 valeurs

In [ ]:
import sys

In [ ]:
def fibonacci():
    """Génère une suite infinie de nombres de Fibonacci."""
    a, b = 0, 1
    while True:
        yield a
        a, b = b, a + b

gen = fibonacci()

In [ ]:
print("Taille du générateur :", sys.getsizeof(gen))

In [ ]:
def fibonacci(n):
    """Génère les n premiers nombres de Fibonacci dans une liste."""
    result = []
    a, b = 0, 1
    for _ in range(n):
        result.append(a)
        a, b = b, a + b
    return result

In [ ]:
fibonacci_100 = fibonacci(100)

In [ ]:
fibonacci_100

In [ ]:

print("Taille de la liste :", sys.getsizeof(fibonacci_100))


### Exemple 3: Pipelines de Coroutines

Créons une pipeline qui reçoit une série de nombres, ne garde que les nombres pairs, les multiplie par deux, puis affiche le résultat.

Ici, Chaque fonction est une coroutine qui reçoit des données via `yield`.

- `send` permet d’envoyer une valeur à la coroutine suivante.

- `next(coroutine)` initialise la coroutine avant de pouvoir lui envoyer des valeurs.

- L'ordre de la pipeline :  filtre les pairs → double → affiche.

In [ ]:
# Étape 1 : filtre les nombres pairs
def filtre_pairs(etape_suivant):
    while True:
        nombre = (yield)
        if nombre % 2 == 0:
            etape_suivant.send(nombre)

In [ ]:
# Étape 2 : double les nombres reçus
def double(etape_suivant):
    while True:
        nombre = (yield)
        etape_suivant.send(nombre * 2)


In [ ]:
# Étape 3 : affiche les nombres
def affiche():
    while True:
        nombre = (yield)
        print(f"Résultat: {nombre}")

In [ ]:
# Création des coroutines
etape_affiche = affiche()
next(etape_affiche)  # initialisation

etape_double = double(etape_affiche)
next(etape_double)  # initialisation

etape_filtre = filtre_pairs(etape_double)
next(etape_filtre)  # initialisation


In [ ]:
# Envoi de données dans la pipeline
for i in range(1, 11):
    etape_filtre.send(i)

### Exemple 4: Programme de "Gestion d'état"

In [ ]:
def gestion_compteur():
    compteur = 0
    while True:
        commande = (yield compteur)  # renvoie l'état actuel et attend une commande
        if commande == "incr":
            compteur += 1
        elif commande == "decr":
            compteur -= 1
        elif commande == "reset":
            compteur = 0
        elif commande is None:
            break  # arrêt de la coroutine

In [ ]:
# Création et initialisation de la coroutine
coro = gestion_compteur()

print(coro.send(None))  # initialisation, affiche 0

In [ ]:
# Envoi de commandes
print(coro.send("incr"))
print(coro.send("incr"))
print(coro.send("decr"))
print(coro.send("reset")) 


# Exercice: Simulateur de pipeline de traitement de données

### Étape 0 - Ouvrir un fichier de données csv
Créer une fonction `read_csv_values` qui retourne un générateur permettant de lire chaque ligne d'un fichier csv

In [ ]:
import csv

path = "data/exercice_generateur.csv"

# >>> À COMPLÉTER <<<
def read_csv_values(path):
    pass

### Correction

In [ ]:
import csv

def read_csv_values(path):
    with open(path, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            # On yield *directement la valeur brute* du CSV
            yield row["value"]

In [ ]:
path = "data/exercice_generateur.csv"

In [ ]:
raw_stream = read_csv_values(path)

In [ ]:
next(raw_stream)

### Étape 1 - Nettoyage simple

Créer un générateur dont le job est de nettoyer un dataset en :

- éliminant les valeurs non convertibles en float
- convertissant les valeurs valides en float

In [ ]:
# >>> À COMPLÉTER <<<
def clean_values(values):
    pass


### Correction

In [ ]:

def clean_values(values):
    for x in values:
        try:
            yield float(x)
        except (TypeError, ValueError):
            continue


## Étape 2 - Data validation

A présent, nous voulons un générateur qui :

- filtre les valeurs invalides selon la fonction `is_valid()` fournie
- convertit en float
- arrondie a 2 décimales
- yield les valeurs finales

In [ ]:
def is_valid(x):
    """Considérons qu'une valeur est valide si elle n'est pas négative."""
    return x >= 0

# >>> À COMPLÉTER <<<
def check_and_filter(values):
    pass



#### Correction

In [ ]:
def is_valid(x):
    """Considérons qu'une valeur est valide si elle n'est pas négative."""
    return x >= 0

def check_and_filter(values):
    for x in values:
        try:
            v = float(x)
        except (TypeError, ValueError):
            continue

        if is_valid(v):
            yield round(v, 2)



### Étape 3 - Normalisation

Créer un génerateur permettant de trouver le minimum et le maximum de toute la série de valeur, afin de normaliser les données avec l'équation : $x = \frac{x - min}{max - min}$

In [ ]:
# >>> À COMPLÉTER <<<
def scaled_values(values, min_v, max_v):
    """Applique un min-max scaling."""
    pass


# >>> À COMPLÉTER <<<
def compute_minmax(path):
    """Parcourt le CSV en streaming pour extraire min/max sans charger en RAM."""
    pass

#### Correction

In [ ]:
def scaled_values(values, min_v, max_v):
    """Applique un min-max scaling."""
    rng = max_v - min_v
    for x in values:
        # x est déjà un float à ce stade
        if 0 <= x <= 10000:
            yield (x - min_v) / rng

def compute_minmax(path):
    """Parcourt le CSV en streaming pour extraire min/max sans charger en RAM."""
    raw_stream = read_csv_values(path)

    min_v = float("inf")
    max_v = float("-inf")

    for x in raw_stream:
        try:
            v = float(x)
        except:
            continue

        if v < min_v: min_v = v
        if v > max_v: max_v = v

    return min_v, max_v

### Création de la pipeline finale

mettre a la chaine toutes ces opérations afin de créer un générateur final, nommé `full_pipeline` dont le job est de traiter les données les unes apres les autres.

In [ ]:
# >>> A COMPLETER
def full_pipeline(csv_path):
    pass

#### Solution : 

In [ ]:
def full_pipeline(csv_path):
    
    raw_stream = read_csv_values(csv_path)
    step1 = clean_values(raw_stream)
    step2 = check_and_filter(step1)
    min_v, max_v = compute_minmax(csv_path)
    step3 = scaled_values(step2, min_v=min_v, max_v=max_v)

    # La pipeline yield les valeurs finales
    for v in step3:
        yield v


In [ ]:
FILE_PATH = 'data/exercice_generateur.csv'
generator = full_pipeline(FILE_PATH)

In [ ]:
next(generator)

## Étape Bonus ! Affichage des résultats

Améliorer le code avec une (et une seule) fonction supplémentaire vous permettant d'afficher les résultats de chaque étape de la pipeline.

In [ ]:
def ma_super_fonction():
    pass

#### Correction

In [ ]:
from functools import wraps

def print_value(func):
    """Décorateur pour afficher les valeurs issues de chaque Block."""
    @wraps(func)
    def wrapper(*args, **kwargs):
        for v in func(*args, **kwargs):
            print(f"{func.__name__} -> valeur : {v}")
            yield v
    return wrapper

In [ ]:

def read_csv_values(path):
    with open(path, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            # On yield *directement la valeur brute* du CSV
            yield row["value"]


@print_value
def clean_values(values):
    for x in values:
        try:
            yield float(x)
        except (TypeError, ValueError):
            continue


@print_value
def check_and_filter(values):
    for x in values:
        try:
            v = float(x)
        except (TypeError, ValueError):
            continue

        if is_valid(v):
            yield round(v, 2)


@print_value
def scaled_values(values, min_v, max_v):
    """Applique un min-max scaling."""
    rng = max_v - min_v
    for x in values:
        # x est déjà un float à ce stade
        if 0 <= x <= 1000000:
            yield (x - min_v) / rng


In [ ]:
def full_pipeline(csv_path):
    
    raw_stream = read_csv_values(csv_path)
    step1 = clean_values(raw_stream)
    step2 = check_and_filter(step1)
    min_v, max_v = compute_minmax(csv_path)
    step3 = scaled_values(step2, min_v=min_v, max_v=max_v)

    # La pipeline yield les valeurs finales
    for v in step3:
        yield v


In [ ]:
FILE_PATH = 'data/exercice_generateur.csv'
generator = full_pipeline(FILE_PATH)

In [102]:
next(generator)

clean_values -> valeur : 10752.640559173433
check_and_filter -> valeur : 10752.64
scaled_values -> valeur : 0.5347747375103196


0.5347747375103196